# AgentMorph Stage 1 - **Gemma-2-9B**

This notebook runs *only* `Gemma-2-9B` on Colab T4 so each model gets a clean Python process (fresh CUDA allocator, no fragmentation carry-over from prior loads). All 5 per-model notebooks share one `--out-dir` on Drive - the manifest key includes the model id, so there are no file collisions.

**Model size:** mid-large (~7 GB)

**Ordering recommendation:**

1. Start with `Llama-3.2-3B` (canary - smallest, fastest, proves the pipeline end-to-end).
2. Once that shows >= 80% finish rate, run the others.
3. Save `Phi-4` for last - it's the tightest fit on a T4.

**Before you start:**

- Runtime -> Change runtime type -> **T4 GPU**.
- Have a HuggingFace token ready (paste into Section 3).

## 1. GPU sanity check

In [ ]:
import torch
assert torch.cuda.is_available(), "No CUDA visible. Runtime > Change runtime type > T4 GPU."
dev = torch.cuda.get_device_properties(0)
print(f"CUDA OK: {torch.cuda.get_device_name(0)} ({dev.total_memory / 1024**3:.1f} GB VRAM)")

## 2. Clone / pull the repo

In [ ]:
import os
REPO_URL = "https://github.com/Pranamya16/AgentMorph.git"
REPO_DIR = "/content/AgentMorph"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout main && git pull --ff-only
!cd {REPO_DIR} && git log -1 --oneline

## 3. HuggingFace auth

Paste your token, run the cell, then **clear the token from the cell before sharing this notebook.**

In [ ]:
from huggingface_hub import login
HF_TOKEN = "hf_REPLACE_ME"
login(token=HF_TOKEN)

## 4. Mount Drive + install extras

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -e /content/AgentMorph[models,smolagents,langgraph,agentdojo]

## 5. Set the model for this notebook

Only change this line if you know what you're doing - each notebook is named after the model it runs.

In [ ]:
MODEL = "Gemma-2-9B"
print("this notebook will run:", MODEL)

## 6. (Optional) wipe prior `Gemma-2-9B` cells from the shared run

This deletes **only** this model's entries from the shared `manifest.json` + its JSONL files, leaving other models' progress untouched. Run it if a previous run left error-only trajectories for `Gemma-2-9B` that you want to retry after an adapter fix.

In [ ]:
import json, pathlib
RUN_DIR = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline")
manifest_path = RUN_DIR / "manifest.json"
traj_dir = RUN_DIR / "trajectories"

# 1. Prune THIS model's entries from the manifest.
if manifest_path.exists():
    data = json.loads(manifest_path.read_text())
    before = len(data.get("completed", {}))
    data["completed"] = {k: v for k, v in data.get("completed", {}).items() if not k.startswith(MODEL + "|")}
    after = len(data["completed"])
    manifest_path.write_text(json.dumps(data, indent=2))
    print(f"manifest: {before} -> {after} entries (dropped {before - after} for {MODEL})")
else:
    print("no manifest - nothing to prune")

# 2. Rewrite JSONL files to drop THIS model's rows.
if traj_dir.exists():
    for p in traj_dir.glob(f"{MODEL}__*.jsonl"):
        p.unlink()
        print("deleted:", p.name)
else:
    print("no trajectory dir yet")

## 7. Smoke test - `Gemma-2-9B` x native x 3 scenarios

The `native` framework uses our own JSON-in-code-block ReAct loop. If it finishes at least a couple scenarios cleanly, the model loads correctly and the tools + trajectory logger are healthy. Skip straight to Section 8 if you want to go directly to the framework sweep.

In [ ]:
!python -m agentmorph.runner \
    --model {MODEL} --framework native --env ecommerce --n-scenarios 3 \
    --hf-cache-dir /content/drive/MyDrive/AgentMorph/hf_cache \
    --out-dir /content/drive/MyDrive/AgentMorph/runs/stage1_smoke_{MODEL}

## 8. Framework sweep - `Gemma-2-9B` x (smolagents + langgraph) x ecommerce x 20 scenarios

Resumable. Writes into the shared `stage1_baseline/` directory on Drive - safe to run alongside other per-model notebooks.

In [ ]:
!python -m agentmorph.runner \
    --model {MODEL} --framework smolagents --framework langgraph --env ecommerce \
    --hf-cache-dir /content/drive/MyDrive/AgentMorph/hf_cache \
    --out-dir /content/drive/MyDrive/AgentMorph/runs/stage1_baseline

## 9. Diagnostic - full error text for Gemma-2-9B

Dumps the **complete** (non-truncated) content of the first error step per framework. This is what I need to fix the adapters.

In [ ]:
import json, pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/trajectories")
for p in sorted(traj_dir.glob(f"{MODEL}__*.jsonl")):
    print(f"\n======= {p.name} =======")
    shown = False
    for line in p.open():
        t = json.loads(line)
        errs = [s for s in t["steps"] if s["kind"] == "error"]
        if errs and not shown:
            print(f"scenario: {t['scenario_id']}")
            print(f"steps: {len(t['steps'])}  final_answer: {t['final_answer']!r}")
            print("FULL ERROR TEXT:")
            print(errs[0]["content"])
            shown = True
            break
    if not shown:
        print("(no error steps in this file - all trajectories finished cleanly!)")

## 10. Finish-rate for `Gemma-2-9B`

In [ ]:
import json, collections, pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/trajectories")
stats = collections.defaultdict(lambda: [0, 0])
for p in traj_dir.glob(f"{MODEL}__*.jsonl"):
    for line in p.open():
        t = json.loads(line)
        key = (t["model_id"], t["framework_id"])
        stats[key][0] += 1
        if t["final_answer"]:
            stats[key][1] += 1
print(f"=== {MODEL} finish rate ===")
for (m, f), (tot, fin) in sorted(stats.items()):
    pct = 100 * fin / tot if tot else 0
    print(f"  {f:10s}  finished {fin:2d}/{tot:2d}  ({pct:3.0f}%)")

## 11. Peek one trajectory from `Gemma-2-9B`

In [ ]:
import json, pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/trajectories")
for p in sorted(traj_dir.glob(f"{MODEL}__*.jsonl")):
    sample = p.open().readline()
    if not sample:
        continue
    t = json.loads(sample)
    print(f"\n======= {p.name} =======")
    print(f"scenario: {t['scenario_id']}")
    print(f"framework: {t['framework_id']}")
    print(f"steps: {len(t['steps'])}  wall_seconds: {t.get('wall_seconds') or 0:.1f}")
    print(f"final_answer: {t['final_answer']!r}\n")
    for s in t["steps"][:20]:
        tag = s["kind"]
        if tag == "tool_call":
            print(f"  [{tag}] {s['tool_name']}({s['tool_args']})")
        elif tag == "tool_result":
            print(f"  [{tag}] {s['tool_name']} -> {str(s['tool_output'])[:80]}")
        else:
            print(f"  [{tag}] {str(s.get('content', ''))[:200]}")